# Part 2 of the pipeline demo, generating idf from geojson

This notebook demonstrates how to use the EPSM pipeline to:
1. Download building footprints from DTCC

**Authors:** Aaron Qiyu Liu and Sanjay Somanath

In [1]:
# Import required modules
import sys
from pathlib import Path

# Add parent directory to path to import geojson_processor
sys.path.insert(0, str(Path.cwd().parent))


dtcc_output_path = Path.cwd() / "dtcc_output"

from geojson_processor.geojson_to_idf import GeoJSONToIDFConverter

## Step 1: Read the GeoJSON data

Let's first load and inspect the GeoJSON file from the DTCC output.

In [9]:
import json

# Load the GeoJSON file
geojson_path = dtcc_output_path / "city.geojson"

with open(geojson_path, 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)

# Display basic information
print(f"GeoJSON Type: {geojson_data.get('type')}")
print(f"Number of features: {len(geojson_data.get('features', []))}")
print(f"\nFirst feature properties:")
if geojson_data.get('features'):
    first_feature = geojson_data['features'][0]
    print(f"  Geometry type: {first_feature['geometry']['type']}")
    print(f"  Properties: {list(first_feature.get('properties', {}).keys())}")
    
# Check if features have required properties for IDF conversion
print(f"\nSample feature properties:")
for key, value in list(first_feature.get('properties', {}).items())[:5]:
    print(f"  {key}: {value}")

GeoJSON Type: FeatureCollection
Number of features: 244

First feature properties:
  Geometry type: Polygon
  Properties: ['id', 'objektidentitet', 'versiongiltigfran', 'lagesosakerhetplan', 'lagesosakerhethojd', 'ursprunglig_organisation', 'objektversion', 'objekttypnr', 'objekttyp', 'insamlingslage', 'byggnadsnamn1', 'byggnadsnamn2', 'byggnadsnamn3', 'husnummer', 'huvudbyggnad', 'andamal1', 'andamal2', 'andamal3', 'andamal4', 'andamal5', 'ground_height', 'height']

Sample feature properties:
  id: 904353a4-ec8b-4c5c-8969-84ba07b5a627
  objektidentitet: 76c6b9ab-6214-42a4-a90f-d5c883f70adf
  versiongiltigfran: 2011-03-23T09:03:39.491001+00:00
  lagesosakerhetplan: 0.025
  lagesosakerhethojd: 2.5


## Step 2: Initialize the GeoJSONToIDFConverter

The converter requires a working directory where it will save the output IDF files.

In [13]:
# Create output directory for IDF files
idf_output_path = Path.cwd() / "idf_output"
idf_output_path.mkdir(parents=True, exist_ok=True)

# Initialize the converter
converter = GeoJSONToIDFConverter(work_dir=str(idf_output_path))
print(f"Converter initialized with work directory: {converter.work_dir}")

Converter initialized with work directory: c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output


## Step 2a (Optional): Filter Buildings

Before converting to IDF, you can filter buildings based on:
- **Height range**: Remove very short buildings (< 3m) or very tall buildings (> 100m)
- **Floor area**: Remove very small buildings (< 100 m²)

This helps clean the dataset and speeds up simulation.

In [14]:
# Filter buildings based on height and area criteria
filtered_geojson_path = converter.filter_buildings(
    geojson_path=geojson_path,
    filter_height_less_than=3,      # Remove buildings < 3m tall
    filter_height_greater_than=100,  # Remove buildings > 100m tall
    filter_area_less_than=100        # Remove buildings < 100 m²
)

print(f"Filtered GeoJSON saved to: {filtered_geojson_path}")

# Check how many buildings remain
with open(filtered_geojson_path, 'r', encoding='utf-8') as f:
    filtered_data = json.load(f)
print(f"Buildings after filtering: {len(filtered_data.get('features', []))}")

2025-12-12 14:56:43,722 [geojson_processor.geojson_to_idf] [INFO] Filtering buildings from c:\Users\qiyu\EPSM\epsm\backend\notebooks\dtcc_output\city.geojson
2025-12-12 14:56:43,723 [geojson_processor.geojson_to_idf] [INFO] Filters: height 3m - 100m, area >= 100m²
2025-12-12 14:56:43,723 [geojson_processor.geojson_to_idf] [INFO] Filters: height 3m - 100m, area >= 100m²
2025-12-12 14:56:43,780 [pyogrio._io] [INFO] Created 137 records
2025-12-12 14:56:43,780 [geojson_processor.geojson_to_idf] [INFO] Filtered 107 buildings, 137 remaining
2025-12-12 14:56:43,780 [pyogrio._io] [INFO] Created 137 records
2025-12-12 14:56:43,780 [geojson_processor.geojson_to_idf] [INFO] Filtered 107 buildings, 137 remaining
Filtered GeoJSON saved to: c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output\city_filtered.geojson
Buildings after filtering: 137
Filtered GeoJSON saved to: c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output\city_filtered.geojson
Buildings after filtering: 137


## Step 2b (Optional): Enrich GeoJSON with Building Properties

Enrichment adds properties required by Dragonfly/Honeybee:
- Building ID and name
- Number of stories (calculated from height)
- Window-to-wall ratio
- Building type (simulation vs context shading)

**Simulation Bounds (Optional):**
If you specify simulation bounds, only buildings inside will be fully simulated. Buildings outside become "context shading" (affects shadows but aren't simulated internally), which significantly speeds up simulations for large areas.

In [15]:
# Option 1: Enrich ALL buildings (no simulation bounds)
# This simulates all buildings - can be slow for large datasets
enriched_geojson_path = converter.enrich_geojson(
    geojson_path=filtered_geojson_path  # Use filtered data, or geojson_path for original
)

print(f"Enriched GeoJSON saved to: {enriched_geojson_path}")

# Check enriched properties
with open(enriched_geojson_path, 'r', encoding='utf-8') as f:
    enriched_data = json.load(f)
    
if enriched_data.get('features'):
    sample_props = enriched_data['features'][0]['properties']
    print(f"\nEnriched properties (first building):")
    for key in ['id', 'name', 'type', 'maximum_roof_height', 'number_of_stories', 
                'window_to_wall_ratio', 'building_status']:
        if key in sample_props:
            print(f"  {key}: {sample_props[key]}")

2025-12-12 14:56:51,703 [geojson_processor.geojson_to_idf] [INFO] Enriching GeoJSON: c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output\city_filtered.geojson
2025-12-12 14:56:51,703 [geojson_processor.geojson_to_idf] [INFO] No simulation bounds - all buildings will be simulated
2025-12-12 14:56:51,703 [geojson_processor.geojson_to_idf] [INFO] No simulation bounds - all buildings will be simulated
2025-12-12 14:56:51,744 [geojson_processor.geojson_to_idf] [INFO] ✅ Enriched GeoJSON: 137 buildings to simulate, 0 context shades
2025-12-12 14:56:51,744 [geojson_processor.geojson_to_idf] [INFO] Saved to c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output\city_enriched.geojson
2025-12-12 14:56:51,744 [geojson_processor.geojson_to_idf] [INFO] ✅ Enriched GeoJSON: 137 buildings to simulate, 0 context shades
2025-12-12 14:56:51,744 [geojson_processor.geojson_to_idf] [INFO] Saved to c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output\city_enriched.geojson
Enriched GeoJSON saved to: c:\Users\qi

In [16]:
# Option 2: Enrich with simulation bounds (RECOMMENDED for large areas)
# Only buildings inside bounds are fully simulated, others are context shading

# Define simulation bounds in EPSG:3006 (Swedish reference system)
# Example: 500m x 500m area in the center of your dataset
import geopandas as gpd

# Get the centroid of your data
gdf = gpd.read_file(filtered_geojson_path)
gdf_3006 = gdf.to_crs('EPSG:3006')
bounds_3006 = gdf_3006.total_bounds
center_x = (bounds_3006[0] + bounds_3006[2]) / 2
center_y = (bounds_3006[1] + bounds_3006[3]) / 2

# Define a 500m x 500m box around the center
simulation_bounds = {
    'west': center_x - 250,
    'east': center_x + 250,
    'south': center_y - 250,
    'north': center_y + 250
}

print(f"Simulation bounds (EPSG:3006):")
print(f"  West:  {simulation_bounds['west']:.2f}")
print(f"  East:  {simulation_bounds['east']:.2f}")
print(f"  South: {simulation_bounds['south']:.2f}")
print(f"  North: {simulation_bounds['north']:.2f}")

# Enrich with simulation bounds
enriched_with_bounds_path = converter.enrich_geojson(
    geojson_path=filtered_geojson_path,
    simulation_bounds=simulation_bounds
)

print(f"\nEnriched GeoJSON with simulation bounds saved to: {enriched_with_bounds_path}")

# Check how many buildings will be simulated vs context shading
with open(enriched_with_bounds_path, 'r', encoding='utf-8') as f:
    enriched_bounds_data = json.load(f)
    
simulated = sum(1 for f in enriched_bounds_data['features'] 
                if f['properties'].get('building_status') == 'Building')
context = sum(1 for f in enriched_bounds_data['features'] 
              if f['properties'].get('building_status') == 'Existing')

print(f"\nBuildings to simulate: {simulated}")
print(f"Context shading: {context}")
print(f"Total: {simulated + context}")

Simulation bounds (EPSG:3006):
  West:  317997.09
  East:  318497.09
  South: 6399002.45
  North: 6399502.45
2025-12-12 14:57:03,884 [geojson_processor.geojson_to_idf] [INFO] Enriching GeoJSON: c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output\city_filtered.geojson
2025-12-12 14:57:03,886 [geojson_processor.geojson_to_idf] [INFO] Simulation bounds provided: {'west': np.float64(317997.0895), 'east': np.float64(318497.0895), 'south': np.float64(6399002.447999999), 'north': np.float64(6399502.447999999)}
2025-12-12 14:57:03,887 [geojson_processor.geojson_to_idf] [INFO] Buildings inside simulation bounds will be simulated, others will be context shading
2025-12-12 14:57:03,886 [geojson_processor.geojson_to_idf] [INFO] Simulation bounds provided: {'west': np.float64(317997.0895), 'east': np.float64(318497.0895), 'south': np.float64(6399002.447999999), 'north': np.float64(6399502.447999999)}
2025-12-12 14:57:03,887 [geojson_processor.geojson_to_idf] [INFO] Buildings inside simulation boun

## Step 2c: Choose which data to convert

Now that you have multiple versions of the GeoJSON data, choose which one to use for IDF conversion:

| Variable | Description | Use Case |
|----------|-------------|----------|
| `geojson_path` | Original DTCC data | Quick test with all buildings |
| `filtered_geojson_path` | Filtered by height/area | Cleaner dataset, still all simulated |
| `enriched_geojson_path` | Filtered + enriched properties | All buildings simulated with proper properties |
| `enriched_with_bounds_path` | Filtered + enriched + simulation bounds | **RECOMMENDED** - Fastest with context shading |

**Recommendation:** Use `enriched_with_bounds_path` for large datasets to significantly reduce simulation time while maintaining accuracy.

In [ ]:
# Select which GeoJSON to convert
# Uncomment the version you want to use:

input_geojson = geojson_path                    # Option 1: Original data (244 buildings)
# input_geojson = filtered_geojson_path         # Option 2: Filtered data
# input_geojson = enriched_geojson_path         # Option 3: Enriched (all simulated)
# input_geojson = enriched_with_bounds_path     # Option 4: RECOMMENDED (simulation bounds)

print(f"Selected input: {input_geojson.name}")
print(f"File exists: {input_geojson.exists()}")

# Check what you're about to convert
if input_geojson.exists():
    with open(input_geojson, 'r', encoding='utf-8') as f:
        data = json.load(f)
    total_features = len(data.get('features', []))
    print(f"Total buildings: {total_features}")
    
    # If enriched with bounds, show simulation vs context breakdown
    if 'building_status' in data['features'][0].get('properties', {}):
        sim_count = sum(1 for f in data['features'] 
                       if f['properties'].get('building_status') == 'Building')
        ctx_count = sum(1 for f in data['features'] 
                       if f['properties'].get('building_status') == 'Existing')
        print(f"  → Buildings to simulate: {sim_count}")
        print(f"  → Context shading: {ctx_count}")
else:
    print("⚠️  File not found! Run the filtering/enrichment steps first.")

## Step 3: Convert GeoJSON to IDF

Now we'll use the converter to generate IDF files from the GeoJSON data. The conversion includes:
- Building geometry from footprints
- Floor heights and building volumes
- Window assignments
- HVAC system templates
- Design day creation for sizing

**Key parameters:**
- `use_multiplier`: Use floor multipliers for repeated floors (faster simulation)
- `timestep`: Simulation timesteps per hour (1 = fastest, 6 = detailed)
- `do_zone_sizing` and `do_system_sizing`: Enable HVAC sizing calculations
- `winter_design_temp` and `summer_design_temp`: Design day temperatures for HVAC sizing

In [17]:
# Convert GeoJSON to IDF
# Note: This may take some time depending on the number of buildings

# Choose which GeoJSON to convert:
# - geojson_path: Original data (all buildings)
# - filtered_geojson_path: Filtered by height/area
# - enriched_geojson_path: Filtered + enriched (all simulated)
# - enriched_with_bounds_path: Filtered + enriched with simulation bounds (RECOMMENDED)

# Use the enriched data if available, otherwise use original
input_geojson = geojson_path  # Change this to use filtered/enriched versions
# input_geojson = enriched_with_bounds_path  # Uncomment after running enrichment

idf_path, gbxml_path = converter.convert_to_idf(
    geojson_path=input_geojson,
    use_multiplier=True,          # Use floor multipliers for faster simulation
    timestep=1,                    # 1 timestep per hour (fastest)
    do_zone_sizing=False,          # Skip zone sizing for speed
    do_system_sizing=False,        # Skip system sizing for speed
    winter_design_temp=-10,        # Winter design temperature (°C)
    summer_design_temp=30,         # Summer design temperature (°C)
    all_polygons_to_buildings=True # Treat all polygons as buildings (needed for DTCC data)
)

print(f"\n✓ Conversion completed successfully!")
print(f"IDF file: {idf_path}")
print(f"GBXML file: {gbxml_path}")
print(f"\nOutput files saved to: {idf_output_path}")

2025-12-12 14:57:28,731 [geojson_processor.geojson_to_idf] [INFO] Converting GeoJSON to IDF: c:\Users\qiyu\EPSM\epsm\backend\notebooks\dtcc_output\city.geojson
2025-12-12 14:57:28,740 [geojson_processor.geojson_to_idf] [INFO] Calculating bounding box centroid...
2025-12-12 14:57:28,740 [geojson_processor.geojson_to_idf] [INFO] Calculating bounding box centroid...
2025-12-12 14:57:28,741 [geojson_processor.geojson_to_idf] [INFO] Centroid: lon=11.950129, lat=57.698632
2025-12-12 14:57:28,741 [geojson_processor.geojson_to_idf] [INFO] Centroid: lon=11.950129, lat=57.698632
2025-12-12 14:57:28,742 [geojson_processor.geojson_to_idf] [INFO] Creating Dragonfly model...
2025-12-12 14:57:28,742 [geojson_processor.geojson_to_idf] [INFO] Creating Dragonfly model...
2025-12-12 14:57:29,010 [geojson_processor.geojson_to_idf] [INFO] Adjusting building properties...
2025-12-12 14:57:29,010 [geojson_processor.geojson_to_idf] [INFO] Adjusting building properties...
2025-12-12 14:57:29,109 [geojson_proce

### Note on `all_polygons_to_buildings` Parameter

The `all_polygons_to_buildings=True` parameter is important for DTCC data because:
- DTCC GeoJSON files don't use standard building property names (like "building", "building:height")
- Instead, they use Swedish property names ("andamal", "byggnadsnamn", etc.)
- Setting this to `True` treats all polygon features as buildings
- The converter will extract height and other properties from the DTCC schema

## Step 4: Inspect the generated IDF file

Let's check the contents of the IDF output directory and display some basic information about the generated file.

In [18]:
# List all files in the output directory
print("Files in output directory:")
for file in idf_output_path.iterdir():
    file_size = file.stat().st_size / 1024  # Size in KB
    print(f"  - {file.name} ({file_size:.1f} KB)")

# Read and display a preview of the IDF file
if idf_path.exists():
    print(f"\n--- Preview of {idf_path.name} (first 30 lines) ---")
    with open(idf_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for i, line in enumerate(lines[:30], 1):
            print(f"{i:3d}: {line}", end='')
    print(f"\n... (Total lines: {len(lines)})")
else:
    print("IDF file not found!")

Files in output directory:
  - city.dfjson (1527.5 KB)
  - city.idf (2951.8 KB)
  - city_enriched.geojson (583.7 KB)
  - city_filtered.geojson (209.6 KB)

--- Preview of city.idf (first 30 lines) ---
  1: !-   ==========================================
  2: !-   =========  SIMULATION PARAMETERS =========
  3: !-   ==========================================
  4: 
  5: 
  6: OutputControl:Table:Style,
  7:  CommaAndHTML,             !- column separator
  8:  None;                     !- unit conversion
  9: 
 10: Output:Variable,
 11:  *,                        !- key value
 12:  Baseboard Electricity Energy, !- name
 13:  Hourly;                   !- frequency
 14: 
 15: Output:Variable,
 16:  *,                        !- key value
 17:  Boiler Electricity Energy, !- name
 18:  Hourly;                   !- frequency
 19: 
 20: Output:Variable,
 21:  *,                        !- key value
 22:  Boiler NaturalGas Energy, !- name
 23:  Hourly;                   !- frequency
 24: 
 25: Outp

## Summary: Complete Workflow

Here's the recommended workflow for processing DTCC data:

### **Basic Workflow** (Quick, all buildings simulated)
1. ✅ Load original GeoJSON from DTCC
2. ✅ Initialize converter
3. ✅ Convert directly to IDF with `all_polygons_to_buildings=True`

### **Optimized Workflow** (Recommended for large datasets)
1. ✅ Load original GeoJSON from DTCC
2. ✅ Initialize converter
3. ✅ **Filter buildings** by height and area (Step 2a)
4. ✅ **Enrich GeoJSON** with simulation bounds (Step 2b, Option 2)
5. ✅ Convert enriched data to IDF (Step 3)

**Why use filtering and enrichment?**
- **Filtering**: Removes invalid/problematic buildings, speeds up processing
- **Enrichment**: Adds required properties, enables simulation bounds
- **Simulation bounds**: Only simulates buildings in area of interest, rest are context shading
- **Result**: Much faster simulation with accurate solar shading from surrounding buildings

**Performance comparison:**
- 244 buildings fully simulated: ~2-4 hours
- 50 buildings simulated + 194 context: ~30-60 minutes